# Stumpff functions $C(z)$ and $S(z)$

**Goal:** define two little functions of one variable $z$ that secretly encode *which kind of orbit* you have — ellipse, parabola, or hyperbola — through the **sign of $z$**. They are the building block that lets a single "universal" Kepler equation (Algorithm 3.3) replace the separate 3.1 (ellipse) and 3.2 (hyperbola).

**Code (answer key):** [`stumpC`, `stumpS`](../curtis/algorithms/stumpff.py) · **Book:** §3.7, Appendix D.4 · Equations 3.49 ($S$) and 3.50 ($C$)

There is no worked-example number here — these are a tool. We will verify them by their **defining series** and by **smoothness through $z=0$** instead.

## Read first

| Symbol | Meaning |
|---|---|
| $z$ | the universal-variable argument $z = \alpha\,\chi^2$ (you will meet $\alpha=1/a$ and $\chi$ in 3.3) |
| sign of $z$ | $z>0$ → ellipse · $z=0$ → parabola · $z<0$ → hyperbola |
| $C(z)$ | Stumpff cosine-like function (Eq 3.50) |
| $S(z)$ | Stumpff sine-like function (Eq 3.49) |

$$C(z)=\begin{cases}\dfrac{1-\cos\sqrt{z}}{z} & z>0\\[6pt]\dfrac{\cosh\sqrt{-z}-1}{-z} & z<0\\[6pt]\tfrac12 & z=0\end{cases}\qquad S(z)=\begin{cases}\dfrac{\sqrt{z}-\sin\sqrt{z}}{(\sqrt{z})^3} & z>0\\[6pt]\dfrac{\sinh\sqrt{-z}-\sqrt{-z}}{(\sqrt{-z})^3} & z<0\\[6pt]\tfrac16 & z=0\end{cases}$$

## The picture (one function, three faces)

3.1 and 3.2 needed *two different equations* because an ellipse uses $\sin/\cos$ and a hyperbola uses $\sinh/\cosh$. That split is annoying — a near-parabolic comet ($e\approx1$) sits right at the awkward boundary.

$C(z)$ and $S(z)$ dissolve the split. Each is **one smooth (entire) function** with a single power series valid for *all* $z$; the trig form ($z>0$) and the hyperbolic form ($z<0$) are just two faces of the same function, meeting seamlessly at the parabolic value $z=0$. Feed it $z>0$ and it behaves like an ellipse; $z<0$, like a hyperbola; $z=0$, like a parabola — automatically.

That is what makes Algorithm 3.3 *universal*: write Kepler's equation once in terms of $C(z), S(z)$ and the sign of $z$ does the case-work for you.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# C(z) and S(z) across the three regimes (computed inline here, so the plot
# works before you implement stumpC/stumpS below).
def _stumpff(z):
    z = np.asarray(z, float)
    C = np.full_like(z, 0.5)
    S = np.full_like(z, 1/6)
    pos, neg = z > 0, z < 0
    sp = np.sqrt(z[pos]);  C[pos] = (1 - np.cos(sp))/z[pos];      S[pos] = (sp - np.sin(sp))/sp**3
    sn = np.sqrt(-z[neg]); C[neg] = (np.cosh(sn) - 1)/(-z[neg]);  S[neg] = (np.sinh(sn) - sn)/sn**3
    return C, S

z = np.linspace(-12, 40, 600)
C, S = _stumpff(z)

fig, ax = plt.subplots(figsize=(7.2, 4.6))
ax.axvspan(-12, 0, color='#D4537E', alpha=0.07)     # z < 0 : hyperbola
ax.axvspan(0, 40, color='#1D9E75', alpha=0.07)      # z > 0 : ellipse
ax.axhline(0, color='#c2c0b6', lw=0.8)
ax.axvline(0, color='#888780', lw=1, ls='--')
ax.plot(z, C, color='#378ADD', lw=2, label='C(z)')
ax.plot(z, S, color='#D85A30', lw=2, label='S(z)')
ax.plot(0, 0.5, 'o', color='#378ADD'); ax.plot(0, 1/6, 'o', color='#D85A30')
ax.annotate('C(0) = 1/2', (1.0, 0.52), color='#378ADD')
ax.annotate('S(0) = 1/6', (1.0, 0.10), color='#D85A30')
ax.annotate('hyperbola\n(z < 0)', (-9, 0.78), color='#D4537E', ha='center', fontsize=10)
ax.annotate('ellipse (z > 0)', (22, 0.78), color='#1D9E75', ha='center', fontsize=10)
ax.set_ylim(-0.05, 0.9)
ax.set_xlabel(r'$z = \alpha\,\chi^2$')
ax.set_title('C(z) and S(z) pass smoothly through z = 0 (the parabola)')
ax.legend(loc='center right')
plt.show()

## See it — there is no kink at $z=0$

The two curves glide through $z=0$ with no corner: the $z>0$ (trig) branch and the $z<0$ (hyperbolic) branch are the *same* analytic function continued across zero. At the crossing they take the finite values $C(0)=\tfrac12$ and $S(0)=\tfrac16$ — the parabolic case is just a point on a smooth curve, not a special case to handle separately.

- For $z>0$ (ellipse) both decay and gently oscillate (that is the $\cos/\sin$ showing through).
- For $z<0$ (hyperbola) both climb (the $\cosh/\sinh$).
- The shaded regions are only labels — the function itself does not "switch"; *you* just evaluate a different closed form to avoid taking the square root of a negative number in code.

## Where these come from

### One series each (this is the real definition)
$$C(z)=\sum_{k=0}^{\infty}\frac{(-z)^k}{(2k+2)!}=\frac1{2!}-\frac{z}{4!}+\frac{z^2}{6!}-\cdots,\qquad
S(z)=\sum_{k=0}^{\infty}\frac{(-z)^k}{(2k+3)!}=\frac1{3!}-\frac{z}{5!}+\frac{z^2}{7!}-\cdots$$
These converge for **every** $z$ (entire functions) — that is *why* $C$ and $S$ are smooth everywhere. Setting $z=0$ keeps only the first term: $C(0)=\tfrac1{2!}=\tfrac12,\ S(0)=\tfrac1{3!}=\tfrac16$.

### The closed forms fall out of the trig series
Let $\phi=\sqrt{z}$ for $z>0$. From $\cos\phi = 1-\frac{\phi^2}{2!}+\frac{\phi^4}{4!}-\cdots$,
$$1-\cos\phi=\frac{\phi^2}{2!}-\frac{\phi^4}{4!}+\cdots \ \Rightarrow\ \frac{1-\cos\phi}{\phi^2}=\sum_k\frac{(-1)^k\phi^{2k}}{(2k+2)!}=\sum_k\frac{(-z)^k}{(2k+2)!}=C(z).$$
So $C(z)=\dfrac{1-\cos\sqrt z}{z}$. The same step on $\phi-\sin\phi$ gives $S(z)=\dfrac{\sqrt z-\sin\sqrt z}{(\sqrt z)^3}$.

### And the hyperbolic forms are the *same* series for $z<0$
For $z<0$ put $\phi=\sqrt{-z}$. Since $\cos(i\phi)=\cosh\phi$ and $\sin(i\phi)=i\sinh\phi$, the identical series now sum to $C(z)=\dfrac{\cosh\sqrt{-z}-1}{-z}$ and $S(z)=\dfrac{\sinh\sqrt{-z}-\sqrt{-z}}{(\sqrt{-z})^3}$. **Same function, continued past zero** — not a new definition.

### Why the sign of $z$ *is* the conic
In 3.3 you will have $z=\alpha\chi^2$ with $\alpha=1/a$. Ellipse $a>0\Rightarrow z>0$; hyperbola $a<0\Rightarrow z<0$; parabola $a\to\infty\Rightarrow z\to0$. So one Kepler equation written with $C(z),S(z)$ adapts to the orbit type on its own.

## Step by step (in code order)

Each function is three branches guarding against $\sqrt{\text{negative}}$:

**`stumpC(z)`** — $z>0:\ (1-\cos\sqrt z)/z$; $\;z<0:\ (\cosh\sqrt{-z}-1)/(-z)$; $\;z=0:\ 1/2$.

**`stumpS(z)`** — $z>0:\ (\sqrt z-\sin\sqrt z)/(\sqrt z)^3$; $\;z<0:\ (\sinh\sqrt{-z}-\sqrt{-z})/(\sqrt{-z})^3$; $\;z=0:\ 1/6$.

**↓ Now type both functions below.** Plain `math` is fine (scalar `z`); use `if/elif/else` on the sign of `z`. Answer key linked above; peek only after you try.

In [ ]:
import math

def stumpC(z):
    """Stumpff function C(z)  (Eq 3.50)."""
    # z > 0:  (1 - cos(sqrt(z))) / z
    # z < 0:  (cosh(sqrt(-z)) - 1) / (-z)
    # z = 0:  1/2
    raise NotImplementedError("fill in the three branches, then delete this line")


def stumpS(z):
    """Stumpff function S(z)  (Eq 3.49)."""
    # z > 0:  (sqrt(z) - sin(sqrt(z))) / sqrt(z)**3
    # z < 0:  (sinh(sqrt(-z)) - sqrt(-z)) / sqrt(-z)**3
    # z = 0:  1/6
    raise NotImplementedError("fill in the three branches, then delete this line")

## Verify — values, smoothness, and the series

No book example here, so we check the three things that *define* these functions: the value at $z=0$, that the branches stitch smoothly across $z=0$, and that they match the power series for small $z$. Run once both functions are typed.

In [ ]:
# 1. Values at z = 0 (the parabola limit)
print(f"C(0) = {stumpC(0)}   (expect 0.5)")
print(f"S(0) = {stumpS(0)}   (expect {1/6})")
assert abs(stumpC(0) - 0.5) < 1e-12
assert abs(stumpS(0) - 1/6) < 1e-12

# 2. The two branches stitch smoothly across z = 0 (no kink)
for z in (1e-6, -1e-6):
    assert abs(stumpC(z) - 0.5) < 1e-6
    assert abs(stumpS(z) - 1/6) < 1e-6

# 3. Match the defining power series near 0
#    C = 1/2 - z/24 + z^2/720 - ... ;   S = 1/6 - z/120 + z^2/5040 - ...
for z in (0.3, -0.3):
    assert abs(stumpC(z) - (1/2 - z/24 + z**2/720)) < 1e-5
    assert abs(stumpS(z) - (1/6 - z/120 + z**2/5040)) < 1e-5

print("\nAll checks passed ✔")

## What this confirms

- $C(z)$ and $S(z)$ are **one smooth function each**, not a pile of cases — the `if/elif/else` only avoids $\sqrt{\text{negative}}$ in code; mathematically nothing special happens at $z=0$.
- The **sign of $z$ carries the orbit type**, which is exactly what lets a single equation cover ellipse, parabola and hyperbola.
- The value at the parabola ($C=\tfrac12,\ S=\tfrac16$) is just the first term of the series.

**Next:** Algorithm 3.3 (`kepler_U`) — the **universal Kepler's equation**, written with $C(z)$ and $S(z)$, which finds the universal anomaly $\chi$ for *any* orbit and supersedes both 3.1 and 3.2 at once.